In [ ]:

# ── 1.2  Core imports 
import re
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud

# NLP
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Machine learning
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay
)

# ── 1.3  Download NLTK corpora 
for resource in ["punkt", "punkt_tab", "stopwords",
                 "vader_lexicon", "averaged_perceptron_tagger"]:
    nltk.download(resource, quiet=True)

# ── 1.4  Global settings 
RANDOM_STATE = 42                          # reproducibility seed throughout
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "figure.figsize": (10, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
})

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:.4f}".format)

sia        = SentimentIntensityAnalyzer()   # VADER — initialised once, reused later
STOP_WORDS = set(stopwords.words("english"))

print("=" * 65)

print("=" * 65)
print(f"  Random state : {RANDOM_STATE}")
print(f"  Stopwords    : {len(STOP_WORDS)} English words loaded")
print(f"  VADER        : ready")


# =============================================================================
# STEP 2 — DATA LOADING & INSPECTION


print("\n" + "=" * 65)
print("   Data Loading & Inspection")
print("=" * 65)

# ── 2.1  Load raw CSV ─────────────────────────────────────────────────────────


FILEPATH    = "Reviews.csv"   # ← change path here if needed
SAMPLE_SIZE = 50000          # sample for tractable runtime; set None for full data

print(f"\n  Loading: {FILEPATH}")
df_raw = pd.read_csv(FILEPATH)
print(f"  Raw shape       : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")

# ── 2.2  Basic structure ──────────────────────────────────────────────────────
print("\n  ── Column names & dtypes ──")
print(df_raw.dtypes.to_string())

print("\n  ── First 3 rows (selected columns) ──")
preview_cols = ["Id", "ProductId", "Score",
                "HelpfulnessNumerator", "HelpfulnessDenominator",
                "Summary", "Text"]
print(df_raw[preview_cols].head(3).to_string(index=False))

# ── 2.3  Missing values ───────────────────────────────────────────────────────
print("\n  ── Missing values per column ──")
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({"Missing Count": missing,
                            "Missing %": missing_pct})
missing_df = missing_df[missing_df["Missing Count"] > 0]
if missing_df.empty:
    print("  No missing values found.")
else:
    print(missing_df.to_string())

# ── 2.4  Basic descriptive statistics ────────────────────────────────────────
print("\n  ── Descriptive statistics (numeric columns) ──")
print(df_raw[["Score", "HelpfulnessNumerator",
              "HelpfulnessDenominator"]].describe().to_string())

# ── 2.5  Star rating distribution ────────────────────────────────────────────
print("\n  ── Star rating distribution ──")
rating_counts = df_raw["Score"].value_counts().sort_index()
for star, count in rating_counts.items():
    bar = "█" * int(count / rating_counts.max() * 30)
    print(f"  {star}★  {bar}  {count:,}")

# ── 2.6  Remove duplicates on review text ────────────────────────────────────
n_before = len(df_raw)
df_raw = df_raw.drop_duplicates(subset="Text")
n_after = len(df_raw)
print(f"\n  Duplicates removed  : {n_before - n_after:,}")
print(f"  Rows after dedup    : {n_after:,}")

# ── 2.7  Sample for computational tractability ───────────────────────────────
if SAMPLE_SIZE and len(df_raw) > SAMPLE_SIZE:
    df = df_raw.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"  Sampled to          : {len(df):,} rows (random_state={RANDOM_STATE})")
else:
    df = df_raw.reset_index(drop=True)
    print(f"  Using full dataset  : {len(df):,} rows")

# ── 2.8  Quick text length check ─────────────────────────────────────────────
df["_text_len"] = df["Text"].fillna("").apply(len)
print(f"\n  Review text length  :")
print(f"    Min    : {df['_text_len'].min()} chars")
print(f"    Median : {df['_text_len'].median():.0f} chars")
print(f"    Max    : {df['_text_len'].max():,} chars")
print(f"    Mean   : {df['_text_len'].mean():.0f} chars")
df = df.drop(columns=["_text_len"])   # temp column — removed before modelling

# ── 2.9  Sample review examples ──────────────────────────────────────────────
print("\n  ── Sample review (first row) ──")
sample = df.iloc[0]
print(f"  Score   : {sample['Score']}")
print(f"  Summary : {sample['Summary']}")
print(f"  Text    : {str(sample['Text'])[:300]}...")
print(f"  Helpful : {sample['HelpfulnessNumerator']} / {sample['HelpfulnessDenominator']}")

print("\n" + "=" * 65)

print("=" * 65)



# =============================================================================
# STEP 3 — DATA CLEANING & LABEL CREATION
# =============================================================================


print("\n" + "=" * 65)
print("  Data Cleaning & Label Creation")
print("=" * 65)

# ── 3.1  Work on a fresh copy so raw data is preserved ───────────────────────
df_clean = df.copy()
print(f"\n  Starting rows : {len(df_clean):,}")

# ── 3.2  Drop rows with missing Text or Summary ───────────────────────────────
before = len(df_clean)
df_clean = df_clean.dropna(subset=["Text", "Summary"])
print(f"  Dropped null Text/Summary : {before - len(df_clean):,} rows")

# ── 3.3  Strip HTML tags that sometimes appear in Amazon reviews ───────────────
#  e.g. "<br />" line breaks embedded in the text field
def clean_text(text):
    """Remove HTML tags, collapse whitespace, strip leading/trailing spaces."""
    text = re.sub(r"<[^>]+>", " ", str(text))   # remove HTML tags
    text = re.sub(r"\s+",     " ", text)          # collapse multiple spaces
    return text.strip()

df_clean["Text"]    = df_clean["Text"].apply(clean_text)
df_clean["Summary"] = df_clean["Summary"].apply(clean_text)
print(f"  HTML tags cleaned from Text & Summary")

# ── 3.4  Remove very short reviews (< 10 words) ───────────────────────────────
# Extremely short reviews carry almost no linguistic signal and are likely
# spam or placeholder text (e.g. "Great!", "Terrible.").
df_clean["word_count_raw"] = df_clean["Text"].apply(lambda x: len(x.split()))
before = len(df_clean)
df_clean = df_clean[df_clean["word_count_raw"] >= 10]
print(f"  Dropped reviews < 10 words : {before - len(df_clean):,} rows")

# ── 3.5  Filter to reviews with ≥ 5 helpfulness votes ────────────────────────
# With fewer than 5 votes the helpfulness ratio is statistically unreliable.
# e.g. 1/1 = 100% but that single vote is not a robust signal.
before = len(df_clean)
df_clean = df_clean[df_clean["HelpfulnessDenominator"] >= 2]
print(f"  Dropped reviews with < 2 votes : {before - len(df_clean):,} rows")
print(f"  Remaining rows : {len(df_clean):,}")

# ── 3.6  Remove impossible helpfulness values ─────────────────────────────────
# Numerator should never exceed denominator — flag and drop if it does.
bad_helpful = df_clean["HelpfulnessNumerator"] > df_clean["HelpfulnessDenominator"]
print(f"  Impossible helpfulness rows (num > denom) : {bad_helpful.sum():,} — dropped")
df_clean = df_clean[~bad_helpful]

# ── 3.7  Compute helpfulness ratio ────────────────────────────────────────────
df_clean["helpfulness_ratio"] = (
    df_clean["HelpfulnessNumerator"] / df_clean["HelpfulnessDenominator"]
)

# ── 3.8  Create binary label ──────────────────────────────────────────────────
# THRESHOLD = 0.6 — majority of voters found the review helpful
HELPFULNESS_THRESHOLD = 0.6

df_clean["helpful"] = (
    df_clean["helpfulness_ratio"] >= HELPFULNESS_THRESHOLD
).astype(int)

label_counts = df_clean["helpful"].value_counts()
total        = len(df_clean)

print(f"\n  ── Helpfulness label distribution (threshold = {HELPFULNESS_THRESHOLD}) ──")
print(f"  Helpful     (1) : {label_counts.get(1, 0):,}  "
      f"({label_counts.get(1,0)/total*100:.1f}%)")
print(f"  Not helpful (0) : {label_counts.get(0, 0):,}  "
      f"({label_counts.get(0,0)/total*100:.1f}%)")

# ── 3.9  Reset index for clean downstream indexing ───────────────────────────
df_clean = df_clean.reset_index(drop=True)

# ── 3.10 Final cleaned dataset summary ───────────────────────────────────────
print(f"\n  ── Final cleaned dataset ──")
print(f"  Shape         : {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print(f"  Columns kept  : {list(df_clean.columns)}")
print(f"  Score range   : {df_clean['Score'].min()} – {df_clean['Score'].max()}")
print(f"  Ratio range   : {df_clean['helpfulness_ratio'].min():.3f} – "
      f"{df_clean['helpfulness_ratio'].max():.3f}")
print(f"  Avg word count: {df_clean['word_count_raw'].mean():.0f} words per review")

# ── 3.11 Quick sanity-check on label vs star rating ──────────────────────────
# We expect helpful reviews to cluster at extreme star ratings
# (strong opinions are more persuasive and voted up more often)
print(f"\n  ── Mean star rating by helpfulness label ──")
print(df_clean.groupby("helpful")["Score"].mean().rename(
    {0: "Not Helpful (0)", 1: "Helpful (1)"}
).to_string())

print("\n" + "=" * 65)
print("   Data cleaned, label created")
print("=" * 65)
print(f"  Clean dataset stored in : df_clean  ({len(df_clean):,} rows)")



# =============================================================================
# STEP 4 — EDA PART 1: DISTRIBUTION PLOTS
# =============================================================================


print("\n" + "=" * 65)
print("   EDA Part 1: Distribution Plots")
print("=" * 65)

# ── 4.1  Figure 1: Helpfulness ratio distribution ────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(
    df_clean["helpfulness_ratio"],
    bins=40,
    color="#4C72B0",
    edgecolor="white",
    linewidth=0.5,
    alpha=0.85
)
ax.axvline(
    HELPFULNESS_THRESHOLD,
    color="#DD4949",
    linewidth=2,
    linestyle="--",
    label=f"Threshold = {HELPFULNESS_THRESHOLD}"
)

# Annotate class counts on each side of the threshold
n_helpful     = (df_clean["helpfulness_ratio"] >= HELPFULNESS_THRESHOLD).sum()
n_not_helpful = len(df_clean) - n_helpful

ax.text(HELPFULNESS_THRESHOLD - 0.03, ax.get_ylim()[1] * 0.85,
        f"Not Helpful\nn={n_not_helpful:,}",
        ha="right", color="#DD4949", fontsize=10, fontweight="bold")
ax.text(HELPFULNESS_THRESHOLD + 0.03, ax.get_ylim()[1] * 0.85,
        f"Helpful\nn={n_helpful:,}",
        ha="left", color="#2ca02c", fontsize=10, fontweight="bold")

ax.set_title("Distribution of Review Helpfulness Ratio", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Helpfulness Ratio  (votes helpful / total votes)")
ax.set_ylabel("Number of Reviews")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("fig1_helpfulness_ratio.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig1_helpfulness_ratio.png")

# ── 4.2  Figure 2: Class balance bar chart ───────────────────────────────────
label_map    = {0: "Not Helpful", 1: "Helpful"}
label_counts = df_clean["helpful"].value_counts().sort_index()
labels       = [label_map[k] for k in label_counts.index]
colors       = ["#DD4949", "#2ca02c"]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
fig.suptitle("Class Balance: Helpful vs Not Helpful Reviews",
             fontsize=14, fontweight="bold")

# Bar chart
bars = axes[0].bar(labels, label_counts.values, color=colors,
                   edgecolor="white", width=0.5)
for bar, val in zip(bars, label_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + label_counts.max() * 0.01,
                 f"{val:,}\n({val/len(df_clean)*100:.1f}%)",
                 ha="center", va="bottom", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].set_title("Review Counts by Class")
axes[0].set_ylim(0, label_counts.max() * 1.18)

# Pie chart
axes[1].pie(
    label_counts.values,
    labels=labels,
    colors=colors,
    autopct="%1.1f%%",
    startangle=140,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    textprops={"fontsize": 12}
)
axes[1].set_title("Class Proportion")

plt.tight_layout()
plt.savefig("fig2_class_balance.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig2_class_balance.png")

# ── 4.3  Figure 3: Star rating distribution ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Star Rating Distribution", fontsize=14, fontweight="bold")

# Overall
overall_counts = df_clean["Score"].value_counts().sort_index()
axes[0].bar(overall_counts.index, overall_counts.values,
            color="#4C72B0", edgecolor="white")
axes[0].set_title("Overall Star Ratings")
axes[0].set_xlabel("Star Rating")
axes[0].set_ylabel("Count")
axes[0].set_xticks([1, 2, 3, 4, 5])
for x, y in zip(overall_counts.index, overall_counts.values):
    axes[0].text(x, y + overall_counts.max()*0.01,
                 f"{y:,}", ha="center", va="bottom", fontsize=9)

# Split by helpfulness label
score_by_label = (
    df_clean.groupby(["Score", "helpful"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "Not Helpful", 1: "Helpful"})
)
score_by_label.plot(
    kind="bar",
    ax=axes[1],
    color=["#DD4949", "#2ca02c"],
    edgecolor="white",
    width=0.7
)
axes[1].set_title("Star Ratings Split by Helpfulness Label")
axes[1].set_xlabel("Star Rating")
axes[1].set_ylabel("Count")
axes[1].set_xticklabels([1, 2, 3, 4, 5], rotation=0)
axes[1].legend(title="Label")

plt.tight_layout()
plt.savefig("fig3_star_ratings.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig3_star_ratings.png")

# ── 4.4  Figure 4: Review text length distribution by label ──────────────────
df_clean["text_word_count"] = df_clean["Text"].apply(lambda x: len(x.split()))

# Cap at 500 words for visual clarity (long tail)
plot_df = df_clean[df_clean["text_word_count"] <= 500].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Review Length Distribution by Helpfulness Label",
             fontsize=14, fontweight="bold")

# KDE plot
for label_val, color, name in [(0, "#DD4949", "Not Helpful"),
                                 (1, "#2ca02c", "Helpful")]:
    subset = plot_df[plot_df["helpful"] == label_val]["text_word_count"]
    subset.plot(kind="kde", ax=axes[0], color=color, linewidth=2, label=name)
axes[0].set_title("Density: Word Count (capped at 500)")
axes[0].set_xlabel("Word Count")
axes[0].set_ylabel("Density")
axes[0].legend()

# Box plot
bp_data = [
    plot_df[plot_df["helpful"] == 0]["text_word_count"].values,
    plot_df[plot_df["helpful"] == 1]["text_word_count"].values
]
bp = axes[1].boxplot(bp_data, patch_artist=True, widths=0.5,
                      medianprops={"color": "white", "linewidth": 2})
for patch, color in zip(bp["boxes"], ["#DD4949", "#2ca02c"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[1].set_xticklabels(["Not Helpful", "Helpful"])
axes[1].set_title("Box Plot: Word Count by Label")
axes[1].set_ylabel("Word Count")

plt.tight_layout()
plt.savefig("fig4_review_length.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig4_review_length.png")

# ── 4.5  Print key EDA statistics ────────────────────────────────────────────
print(f"\n  ── Key EDA statistics ──")
print(f"  Median word count — Not Helpful : "
      f"{df_clean[df_clean['helpful']==0]['text_word_count'].median():.0f} words")
print(f"  Median word count — Helpful     : "
      f"{df_clean[df_clean['helpful']==1]['text_word_count'].median():.0f} words")
print(f"  Most common star rating         : "
      f"{df_clean['Score'].mode()[0]}★ "
      f"({(df_clean['Score']==df_clean['Score'].mode()[0]).mean()*100:.1f}% of reviews)")
print(f"  Mean helpfulness ratio          : "
      f"{df_clean['helpfulness_ratio'].mean():.3f}")

print("\n" + "=" * 65)
print("  STEP 4 COMPLETE — EDA Part 1: Distribution Plots")
print("=" * 65)
print("  Figures saved:")
for f in ["fig1_helpfulness_ratio.png", "fig2_class_balance.png",
          "fig3_star_ratings.png", "fig4_review_length.png"]:
    print(f"    - {f}")
print("  ➡  Next: Step 5 — EDA Part 2: Language Pattern Analysis")


# =============================================================================
# STEP 5 — EDA PART 2: LANGUAGE PATTERN ANALYSIS


print("\n" + "=" * 65)
print("   EDA Part 2: Language Pattern Analysis")
print("=" * 65)

# ── 5.0  Helper: tokenise and remove stopwords ───────────────────────────────
def tokenise_clean(text):
    """Lowercase, tokenise, remove stopwords and non-alpha tokens."""
    tokens = word_tokenize(str(text).lower())
    return [t for t in tokens if t.isalpha() and t not in STOP_WORDS]

# Split corpus by label
helpful_texts     = df_clean[df_clean["helpful"] == 1]["Text"].fillna("").tolist()
not_helpful_texts = df_clean[df_clean["helpful"] == 0]["Text"].fillna("").tolist()

helpful_corpus     = " ".join(helpful_texts)
not_helpful_corpus = " ".join(not_helpful_texts)

print(f"\n  Helpful reviews     : {len(helpful_texts):,}")
print(f"  Not helpful reviews : {len(not_helpful_texts):,}")

# ── 5.1  Figure 5: Word clouds ───────────────────────────────────────────────
print("\n  Generating word clouds...")

def make_wordcloud(text, colormap, title, ax):
    """Generate and plot a word cloud on a given axes object."""
    wc = WordCloud(
        width=800, height=400,
        background_color="white",
        colormap=colormap,
        max_words=120,
        stopwords=STOP_WORDS,
        collocations=False,        # avoid duplicate bigrams
        random_state=RANDOM_STATE
    ).generate(text)
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(title, fontsize=13, fontweight="bold", pad=10)
    ax.axis("off")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Word Clouds: Language of Helpful vs Not Helpful Reviews",
             fontsize=14, fontweight="bold", y=1.01)

make_wordcloud(helpful_corpus,     "Greens",  "✅ Helpful Reviews",     axes[0])
make_wordcloud(not_helpful_corpus, "Reds",    "❌ Not Helpful Reviews", axes[1])

plt.tight_layout()
plt.savefig("fig5_wordclouds.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig5_wordclouds.png")

# ── 5.2  Figure 6: Top 20 word frequencies per class ────────────────────────
print("  Computing word frequencies...")

from collections import Counter

def top_n_words(texts, n=20):
    """Return a Series of the top-n word frequencies across a list of texts."""
    all_tokens = []
    for text in texts:
        all_tokens.extend(tokenise_clean(text))
    counts = Counter(all_tokens)
    return pd.Series(dict(counts.most_common(n)))

top_helpful     = top_n_words(helpful_texts,     n=20)
top_not_helpful = top_n_words(not_helpful_texts, n=20)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Top 20 Most Frequent Words by Helpfulness Label",
             fontsize=14, fontweight="bold")

for ax, top_words, color, title in [
    (axes[0], top_helpful,     "#2ca02c", "✅ Helpful Reviews"),
    (axes[1], top_not_helpful, "#DD4949", "❌ Not Helpful Reviews")
]:
    top_words.sort_values().plot(kind="barh", ax=ax, color=color, alpha=0.85)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Frequency")
    ax.set_ylabel("Word")
    # Add value labels
    for i, (word, val) in enumerate(top_words.sort_values().items()):
        ax.text(val + top_words.max() * 0.005, i,
                f"{val:,}", va="center", fontsize=8)

plt.tight_layout()
plt.savefig("fig6_top_words.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig6_top_words.png")

# ── 5.3  Figure 7: TF-IDF distinguishing words per class ────────────────────

print("  Computing TF-IDF distinctive words...")

from sklearn.feature_extraction.text import TfidfVectorizer

# Fit TF-IDF on all reviews with class label as document group
tfidf_explorer = TfidfVectorizer(
    max_features=10_000,
    stop_words="english",
    ngram_range=(1, 1),
    min_df=1,          # MUST be 1 — this vectoriser fits on only 2 documents
    sublinear_tf=True  # (one per class), so min_df>=2 would raise ValueError
)

# Build per-class pseudo-documents (concatenate all texts per class)
class_docs = [helpful_corpus, not_helpful_corpus]
tfidf_matrix = tfidf_explorer.fit_transform(class_docs)
feature_names = np.array(tfidf_explorer.get_feature_names_out())

# Top words by TF-IDF score for each class
def top_tfidf_words(tfidf_row, feature_names, n=20):
    """Return top-n words by TF-IDF score for a given document row."""
    scores = tfidf_row.toarray().flatten()
    top_idx = scores.argsort()[::-1][:n]
    return pd.Series(scores[top_idx], index=feature_names[top_idx])

top_tfidf_helpful     = top_tfidf_words(tfidf_matrix[0], feature_names, n=20)
top_tfidf_not_helpful = top_tfidf_words(tfidf_matrix[1], feature_names, n=20)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Top 20 Distinctive Words by TF-IDF Score (Class-Level)",
             fontsize=14, fontweight="bold")

for ax, tfidf_words, color, title in [
    (axes[0], top_tfidf_helpful,     "#2ca02c", "✅ Helpful Reviews — Distinctive Words"),
    (axes[1], top_tfidf_not_helpful, "#DD4949", "❌ Not Helpful Reviews — Distinctive Words")
]:
    tfidf_words.sort_values().plot(kind="barh", ax=ax, color=color, alpha=0.85)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("TF-IDF Score")
    ax.set_ylabel("Word")

plt.tight_layout()
plt.savefig("fig7_tfidf_words.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig7_tfidf_words.png")

# ── 5.4  Figure 8: Sentence length & punctuation patterns ───────────────────
# Structural language patterns — longer, more detailed sentences and
# deliberate punctuation use are linked to perceived credibility.
print("  Computing sentence & punctuation patterns...")

def avg_sentence_length(text):
    """Average number of words per sentence."""
    sents = sent_tokenize(str(text))
    if not sents:
        return 0
    return np.mean([len(s.split()) for s in sents])

def exclamation_count(text):
    """Number of exclamation marks — emotional intensity marker."""
    return str(text).count("!")

def question_count(text):
    """Number of question marks — rhetorical engagement marker."""
    return str(text).count("?")

def comma_count(text):
    """Number of commas — proxy for syntactic complexity."""
    return str(text).count(",")

# Compute on a subsample for speed (5,000 per class)
sample_h  = df_clean[df_clean["helpful"] == 1].sample(
    min(5000, (df_clean["helpful"]==1).sum()), random_state=RANDOM_STATE)
sample_nh = df_clean[df_clean["helpful"] == 0].sample(
    min(5000, (df_clean["helpful"]==0).sum()), random_state=RANDOM_STATE)
sample_both = pd.concat([sample_h, sample_nh])

sample_both["avg_sent_len"]      = sample_both["Text"].apply(avg_sentence_length)
sample_both["exclamation_count"] = sample_both["Text"].apply(exclamation_count)
sample_both["question_count"]    = sample_both["Text"].apply(question_count)
sample_both["comma_count"]       = sample_both["Text"].apply(comma_count)

metrics      = ["avg_sent_len", "exclamation_count", "question_count", "comma_count"]
metric_names = ["Avg Sentence\nLength (words)", "Exclamation\nMarks (!)",
                "Question\nMarks (?)", "Commas (,)"]

fig, axes = plt.subplots(1, 4, figsize=(16, 6))
fig.suptitle("Punctuation & Sentence Structure by Helpfulness Label",
             fontsize=14, fontweight="bold")

for ax, metric, name in zip(axes, metrics, metric_names):
    data_h  = sample_both[sample_both["helpful"] == 1][metric]
    data_nh = sample_both[sample_both["helpful"] == 0][metric]

    # Cap outliers at 95th percentile for clean visualisation
    cap = sample_both[metric].quantile(0.95)
    data_h  = data_h.clip(upper=cap)
    data_nh = data_nh.clip(upper=cap)

    ax.boxplot(
        [data_nh, data_h],
        patch_artist=True,
        widths=0.5,
        medianprops={"color": "white", "linewidth": 2}
    )
    if len(bp["boxes"]) >= 2:
        bp["boxes"][0].set_facecolor("#DD4949"); bp["boxes"][0].set_alpha(0.8)
        bp["boxes"][1].set_facecolor("#2ca02c"); bp["boxes"][1].set_alpha(0.8)

    ax.set_xticklabels(["Not\nHelpful", "Helpful"], fontsize=9)
    ax.set_title(name, fontsize=10, fontweight="bold")

    # Annotate with means
    ax.text(1, data_nh.max() * 0.97, f"μ={data_nh.mean():.1f}",
            ha="center", fontsize=8, color="#DD4949")
    ax.text(2, data_h.max()  * 0.97, f"μ={data_h.mean():.1f}",
            ha="center", fontsize=8, color="#2ca02c")

plt.tight_layout()
plt.savefig("fig8_punctuation_patterns.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig8_punctuation_patterns.png")

# ── 5.5  Print language pattern summary stats ────────────────────────────────
print(f"\n  ── Language Pattern Summary ──")
for metric, name in zip(metrics, metric_names):
    mu_h  = sample_both[sample_both["helpful"]==1][metric].mean()
    mu_nh = sample_both[sample_both["helpful"]==0][metric].mean()
    diff  = ((mu_h - mu_nh) / mu_nh * 100) if mu_nh != 0 else 0
    direction = "↑ higher" if diff > 0 else "↓ lower"
    print(f"  {name.replace(chr(10),' '):<30} "
          f"Helpful={mu_h:.2f}  Not={mu_nh:.2f}  "
          f"({direction} in helpful, {abs(diff):.1f}%)")

print("\n" + "=" * 65)
print("   EDA Part 2: Language Pattern Analysis")
print("=" * 65)
print("  Figures saved:")
for f in ["fig5_wordclouds.png", "fig6_top_words.png",
          "fig7_tfidf_words.png", "fig8_punctuation_patterns.png"]:
    print(f"    - {f}")



# =============================================================================
# STEP 6 — LINGUISTIC FEATURE ENGINEERING


print("\n" + "=" * 65)
print(" Linguistic Feature Engineering")
print("=" * 65)

# ── 6.A  Structural features ─────────────────────────────────────────────────

def feat_char_length(text):
    """
    Total character length of the review.
    Longer reviews provide more detail and context — a key persuasion cue.
    """
    return len(str(text))


def feat_word_count(text):
    """
    Total number of words.
    Proxy for review elaborateness; more words = more arguments presented.
    """
    return len(str(text).split())


def feat_sentence_count(text):
    """
    Number of sentences (using NLTK Punkt tokeniser).
    Captures structural complexity beyond simple word count.
    """
    return len(sent_tokenize(str(text)))


def feat_avg_word_length(text):
    """
    Mean character length per word.
    Longer words often indicate more sophisticated or domain-specific vocabulary,
    associated with credibility (Kronrod & Danziger, 2013).
    """
    words = str(text).split()
    if not words:
        return 0.0
    return float(np.mean([len(w) for w in words]))


def feat_avg_sentence_length(text):
    """
    Mean words per sentence.
    Longer sentences can indicate more complex reasoning or elaboration.
    """
    sents = sent_tokenize(str(text))
    if not sents:
        return 0.0
    return float(np.mean([len(s.split()) for s in sents]))


# ── 6.B  Rhetorical features ─────────────────────────────────────────────────

def feat_exclamation_ratio(text):
    """
    Proportion of sentences ending with '!'.
    High exclamation use signals emotional intensity; moderate use may signal
    enthusiasm while overuse can reduce credibility.
    """
    sents = sent_tokenize(str(text))
    if not sents:
        return 0.0
    return sum(1 for s in sents if s.strip().endswith("!")) / len(sents)


def feat_question_ratio(text):
    """
    Proportion of sentences ending with '?'.
    Rhetorical questions signal engagement and reflective writing style.
    """
    sents = sent_tokenize(str(text))
    if not sents:
        return 0.0
    return sum(1 for s in sents if s.strip().endswith("?")) / len(sents)


def feat_first_person_ratio(text):
    """
    Proportion of first-person singular tokens (I, me, my, mine, myself).
    Personal experience narration increases perceived authenticity and
    helpfulness (Willemsen et al., 2011).
    """
    tokens = word_tokenize(str(text).lower())
    fp     = {"i", "me", "my", "mine", "myself"}
    if not tokens:
        return 0.0
    return sum(1 for t in tokens if t in fp) / len(tokens)


def feat_uppercase_ratio(text):
    """
    Ratio of uppercase letters to all letters.
    Extreme values signal shouting (low quality); moderate use signals
    deliberate emphasis.
    """
    letters = [c for c in str(text) if c.isalpha()]
    if not letters:
        return 0.0
    return sum(1 for c in letters if c.isupper()) / len(letters)


# ── 6.C  Specificity features ────────────────────────────────────────────────

def feat_contains_comparison(text):
    """
    Binary flag: review contains comparative language.
    Comparing to alternatives is a persuasion strategy that signals
    informed, experienced opinion (Mudambi & Schuff, 2010).
    """
    pattern = r"\b(better|worse|than|compared|versus|vs\.?|unlike|superior|inferior|prefer)\b"
    return int(bool(re.search(pattern, str(text).lower())))


def feat_specificity_score(text):
    """
    Count of numeric values + measurement units in the review.
    Specific quantitative claims are more credible and persuasive
    (Kronrod & Danziger, 2013). e.g. '2 lbs', '30 days', '5 oz'.
    """
    numbers = len(re.findall(r"\b\d+\.?\d*\b", str(text)))
    units   = len(re.findall(
        r"\b(oz|lb|gram|ml|mg|kg|pound|ounce|inch|cm|mm|"
        r"year|month|week|day|minute|hour|calorie|serving)\b",
        str(text).lower()
    ))
    return numbers + units


# ── 6.D  Credibility features ────────────────────────────────────────────────

def feat_type_token_ratio(text):
    """
    Lexical diversity = unique tokens / total tokens (TTR).
    Higher TTR indicates richer, less repetitive vocabulary — associated
    with more informative and persuasive writing.
    """
    tokens = [t for t in word_tokenize(str(text).lower()) if t.isalpha()]
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)


def feat_hedging_count(text):
    """
    Number of hedging expressions in the review.
    Hedging (maybe, perhaps, might, seems) signals uncertainty and can
    reduce the perceived confidence and persuasiveness of a review.
    """
    hedges = {
        "maybe", "perhaps", "might", "seems", "apparently", "possibly",
        "probably", "somewhat", "sort", "kind", "guess", "suppose",
        "uncertain", "unclear", "unsure", "not sure"
    }
    tokens = set(word_tokenize(str(text).lower()))
    return sum(1 for h in hedges if h in tokens)


# ── 6.E  Sentiment features (VADER) ─────────────────────────────────────────

def feat_sentiment_scores(text):
    """
    VADER sentiment analysis returns four scores:
      compound : overall sentiment (-1 most negative, +1 most positive)
      pos      : proportion of positive language
      neg      : proportion of negative language
      neu      : proportion of neutral language
    VADER is optimised for short social/consumer text (Hutto & Gilbert, 2014).
    Returns a dict — unpacked into separate columns below.
    """
    return sia.polarity_scores(str(text))


# ── 6.1  Apply all features to df_clean ──────────────────────────────────────
print("\n  Applying feature functions to df_clean...")
print("  (This may take 1–2 minutes for 50,000 reviews)\n")

texts = df_clean["Text"].fillna("").astype(str)

# Structural
df_clean["feat_char_length"]        = texts.apply(feat_char_length)
df_clean["feat_word_count"]         = texts.apply(feat_word_count)
df_clean["feat_sentence_count"]     = texts.apply(feat_sentence_count)
df_clean["feat_avg_word_length"]    = texts.apply(feat_avg_word_length)
df_clean["feat_avg_sentence_length"]= texts.apply(feat_avg_sentence_length)
print("  ✅ Structural features done")

# Rhetorical
df_clean["feat_exclamation_ratio"]  = texts.apply(feat_exclamation_ratio)
df_clean["feat_question_ratio"]     = texts.apply(feat_question_ratio)
df_clean["feat_first_person_ratio"] = texts.apply(feat_first_person_ratio)
df_clean["feat_uppercase_ratio"]    = texts.apply(feat_uppercase_ratio)
print("  ✅ Rhetorical features done")

# Specificity
df_clean["feat_contains_comparison"]= texts.apply(feat_contains_comparison)
df_clean["feat_specificity_score"]  = texts.apply(feat_specificity_score)
print("  ✅ Specificity features done")

# Credibility
df_clean["feat_type_token_ratio"]   = texts.apply(feat_type_token_ratio)
df_clean["feat_hedging_count"]      = texts.apply(feat_hedging_count)
print("  ✅ Credibility features done")

# Sentiment (VADER — returns dict, unpack into 4 columns)
sentiments = texts.apply(feat_sentiment_scores)
df_clean["feat_sentiment_compound"] = sentiments.apply(lambda x: x["compound"])
df_clean["feat_sentiment_pos"]      = sentiments.apply(lambda x: x["pos"])
df_clean["feat_sentiment_neg"]      = sentiments.apply(lambda x: x["neg"])
df_clean["feat_sentiment_neu"]      = sentiments.apply(lambda x: x["neu"])
print("  ✅ Sentiment features done")

# Also include the star rating — highly informative signal
df_clean["feat_star_rating"] = df_clean["Score"]
print("  ✅ Star rating feature added")

# ── 6.2  Collect feature column names ────────────────────────────────────────
FEATURE_COLS = [c for c in df_clean.columns if c.startswith("feat_")]
print(f"\n  Total features engineered : {len(FEATURE_COLS)}")
print(f"  Feature names             :")
for i, f in enumerate(FEATURE_COLS, 1):
    print(f"    {i:>2}. {f}")

# ── 6.3  Feature summary statistics by class ─────────────────────────────────
print(f"\n  ── Feature means by helpfulness label ──")
summary = df_clean.groupby("helpful")[FEATURE_COLS].mean().T
summary.columns = ["Not Helpful (0)", "Helpful (1)"]
summary["Δ (Helpful − Not)"] = summary["Helpful (1)"] - summary["Not Helpful (0)"]
summary["Δ %"] = (
    summary["Δ (Helpful − Not)"] / summary["Not Helpful (0)"].replace(0, np.nan) * 100
).round(1)
print(summary.to_string())

# ── 6.4  Check for NaNs or infinite values in features ───────────────────────
nan_counts = df_clean[FEATURE_COLS].isnull().sum()
inf_counts = np.isinf(df_clean[FEATURE_COLS].values).sum(axis=0)
if nan_counts.sum() == 0 and inf_counts.sum() == 0:
    print(f"\n  ✅ No NaN or infinite values in any feature column")
else:
    print(f"\n  ⚠️  NaN values found:\n{nan_counts[nan_counts>0]}")
    # Fill any NaNs with 0 as a safe fallback
    df_clean[FEATURE_COLS] = df_clean[FEATURE_COLS].fillna(0)
    print("     Filled NaNs with 0")

print("\n" + "=" * 65)
print("   Linguistic Feature Engineering")
print("=" * 65)
print(f"  {len(FEATURE_COLS)} features stored in df_clean with prefix 'feat_'")
print("  ➡  Next: Step 7 — EDA Part 3: Feature Analysis & Correlation")


# =============================================================================
# STEP 7 — EDA PART 3: FEATURE ANALYSIS & CORRELATION
# =============================================================================


print("\n" + "=" * 65)
print("   EDA Part 3: Feature Analysis & Correlation")
print("=" * 65)

# ── 7.1  Figure 9: Feature correlations with helpfulness label ───────────────
print("\n  Computing feature–label correlations...")

corr_with_label = (
    df_clean[FEATURE_COLS + ["helpful"]]
    .corr()["helpful"]
    .drop("helpful")
    .sort_values()
)

# Clean display names (remove prefix, underscores → spaces, title case)
clean_names = (
    corr_with_label.index
    .str.replace("feat_", "", regex=False)
    .str.replace("_", " ", regex=False)
    .str.title()
)

colors = ["#DD4949" if v < 0 else "#2ca02c" for v in corr_with_label.values]

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(clean_names, corr_with_label.values, color=colors, alpha=0.85,
               edgecolor="white")

# Annotate each bar with its correlation value
for bar, val in zip(bars, corr_with_label.values):
    x_pos = val + 0.002 if val >= 0 else val - 0.002
    ha    = "left"       if val >= 0 else "right"
    ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
            f"{val:+.3f}", va="center", ha=ha, fontsize=8.5)

ax.axvline(0, color="black", linewidth=0.9, linestyle="-")
ax.set_title("Pearson Correlation of Each Feature with Helpfulness Label",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Pearson r  (positive = more helpful, negative = less helpful)")
ax.set_xlim(corr_with_label.min() - 0.06, corr_with_label.max() + 0.06)

# Add legend patches
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor="#2ca02c", label="Positive correlation"),
                   Patch(facecolor="#DD4949", label="Negative correlation")]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig("fig9_feature_label_correlation.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig9_feature_label_correlation.png")

# ── 7.2  Print top & bottom correlates ───────────────────────────────────────
print(f"\n  ── Top 5 features POSITIVELY correlated with helpfulness ──")
print(corr_with_label.tail(5).rename(
    index=lambda x: x.replace("feat_","").replace("_"," ")
).to_string())

print(f"\n  ── Top 5 features NEGATIVELY correlated with helpfulness ──")
print(corr_with_label.head(5).rename(
    index=lambda x: x.replace("feat_","").replace("_"," ")
).to_string())

# ── 7.3  Figure 10: Violin plots — key features by class ────────────────────
# Choose the 6 most correlated features (3 positive, 3 negative) for display
top3_pos = corr_with_label.tail(3).index.tolist()
top3_neg = corr_with_label.head(3).index.tolist()
key_features = top3_neg + top3_pos   # ordered low → high correlation

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Violin Plots: Top 6 Features by |Correlation| with Helpfulness",
             fontsize=13, fontweight="bold")
axes = axes.flatten()

for ax, feat in zip(axes, key_features):
    # Cap outliers at 98th percentile for visual clarity
    cap   = df_clean[feat].quantile(0.98)
    floor = df_clean[feat].quantile(0.02)
    plot_data = df_clean[[feat, "helpful"]].copy()
    plot_data[feat] = plot_data[feat].clip(lower=floor, upper=cap)

    data_nh = plot_data[plot_data["helpful"] == 0][feat].values
    data_h  = plot_data[plot_data["helpful"] == 1][feat].values

    parts = ax.violinplot([data_nh, data_h], positions=[0, 1],
                          showmedians=True, showextrema=True)

    # Colour the violins
    for i, (pc, color) in enumerate(zip(parts["bodies"],
                                        ["#DD4949", "#2ca02c"])):
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
    parts["cmedians"].set_color("white")
    parts["cmedians"].set_linewidth(2)

    # Clean feature name for title
    title = feat.replace("feat_","").replace("_"," ").title()
    r_val = corr_with_label[feat]
    ax.set_title(f"{title}\n(r = {r_val:+.3f})", fontsize=10, fontweight="bold")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Not Helpful", "Helpful"], fontsize=9)
    ax.set_ylabel("Value")

    # Annotate medians
    ax.text(0, np.median(data_nh), f"med={np.median(data_nh):.2f}",
            ha="center", va="bottom", fontsize=8, color="#DD4949")
    ax.text(1, np.median(data_h),  f"med={np.median(data_h):.2f}",
            ha="center", va="bottom", fontsize=8, color="#2ca02c")

plt.tight_layout()
plt.savefig("fig10_violin_plots.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig10_violin_plots.png")

# ── 7.4  Figure 11: Feature–feature correlation heatmap ─────────────────────
# Check for multicollinearity between features.
# Highly correlated feature pairs (|r| > 0.8) may cause redundancy in
# linear models — important limitation to discuss in the paper.
print("  Computing feature–feature correlation matrix...")

feat_corr_matrix = df_clean[FEATURE_COLS].corr()

# Clean axis labels
clean_feat_labels = (
    pd.Index(FEATURE_COLS)
    .str.replace("feat_", "", regex=False)
    .str.replace("_", "\n", regex=False)
)

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(feat_corr_matrix, dtype=bool), k=1)  # upper triangle

sns.heatmap(
    feat_corr_matrix,
    mask=mask,               # show lower triangle only (avoids duplication)
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.4,
    linecolor="white",
    annot_kws={"size": 7},
    ax=ax,
    square=True,
    cbar_kws={"shrink": 0.6, "label": "Pearson r"}
)

ax.set_xticklabels(clean_feat_labels, fontsize=8, rotation=45, ha="right")
ax.set_yticklabels(clean_feat_labels, fontsize=8, rotation=0)
ax.set_title("Feature–Feature Correlation Matrix\n"
             "(Lower Triangle | Red = negative, Green = positive)",
             fontsize=13, fontweight="bold", pad=14)

plt.tight_layout()
plt.savefig("fig11_feature_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig11_feature_correlation_heatmap.png")

# ── 7.5  Figure 12: Scatter — word count vs helpfulness ratio ────────────────
# A focused scatter plot showing one of the strongest structural signals.
fig, ax = plt.subplots(figsize=(10, 6))

# Sample 3,000 points to avoid overplotting
scatter_sample = df_clean.sample(min(3000, len(df_clean)),
                                 random_state=RANDOM_STATE)
scatter_colors = scatter_sample["helpful"].map({0: "#DD4949", 1: "#2ca02c"})

ax.scatter(
    scatter_sample["feat_word_count"].clip(upper=600),
    scatter_sample["helpfulness_ratio"],
    c=scatter_colors, alpha=0.35, s=18, edgecolors="none"
)

# Add a loess-style trend line using a rolling mean
trend_df = scatter_sample[["feat_word_count", "helpfulness_ratio"]].copy()
trend_df = trend_df[trend_df["feat_word_count"] <= 600].sort_values("feat_word_count")
trend_df["rolling_mean"] = trend_df["helpfulness_ratio"].rolling(150, min_periods=30).mean()
ax.plot(trend_df["feat_word_count"], trend_df["rolling_mean"],
        color="#333333", linewidth=2, linestyle="--", label="Rolling mean trend")

ax.axhline(HELPFULNESS_THRESHOLD, color="navy", linewidth=1.2,
           linestyle=":", label=f"Helpful threshold ({HELPFULNESS_THRESHOLD})")

legend_elements = [
    Patch(facecolor="#2ca02c", alpha=0.7, label="Helpful (1)"),
    Patch(facecolor="#DD4949", alpha=0.7, label="Not Helpful (0)"),
    plt.Line2D([0], [0], color="#333333", linewidth=2,
               linestyle="--", label="Rolling mean trend"),
    plt.Line2D([0], [0], color="navy", linewidth=1.2,
               linestyle=":", label=f"Threshold ({HELPFULNESS_THRESHOLD})")
]
ax.legend(handles=legend_elements, fontsize=9)
ax.set_title("Review Word Count vs Helpfulness Ratio\n"
             "(word count capped at 600 for clarity)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Word Count")
ax.set_ylabel("Helpfulness Ratio")

plt.tight_layout()
plt.savefig("fig12_wordcount_vs_helpfulness.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig12_wordcount_vs_helpfulness.png")

# ── 7.6  Flag high multicollinearity pairs ───────────────────────────────────
print(f"\n  ── Highly correlated feature pairs (|r| > 0.75) ──")
high_corr_pairs = []
for i in range(len(FEATURE_COLS)):
    for j in range(i + 1, len(FEATURE_COLS)):
        r = feat_corr_matrix.iloc[i, j]
        if abs(r) > 0.75:
            high_corr_pairs.append((
                FEATURE_COLS[i].replace("feat_",""),
                FEATURE_COLS[j].replace("feat_",""),
                round(r, 3)
            ))

if high_corr_pairs:
    for f1, f2, r in sorted(high_corr_pairs, key=lambda x: -abs(x[2])):
        print(f"  {f1:<30} ↔  {f2:<30}  r = {r:+.3f}")
    print(f"\n  ⚠️  Note: {len(high_corr_pairs)} high-correlation pairs found.")
    print("  These will be flagged as a limitation in the paper.")
else:
    print("  No pairs with |r| > 0.75 found — low multicollinearity.")

print("\n" + "=" * 65)
print("  STEP 7 COMPLETE — EDA Part 3: Feature Analysis")
print("=" * 65)
print("  Figures saved:")
for f in ["fig9_feature_label_correlation.png", "fig10_violin_plots.png",
          "fig11_feature_correlation_heatmap.png",
          "fig12_wordcount_vs_helpfulness.png"]:
    print(f"    - {f}")
print("  ➡  Next: Step 8 — Model 1: Logistic Regression (Linguistic Features)")


# =============================================================================
# STEP 8 — MODEL 1: LOGISTIC REGRESSION (LINGUISTIC FEATURES ONLY)
# =============================================================================


print("\n" + "=" * 65)
print("   Model 1: Logistic Regression (Linguistic Features Only)")
print("=" * 65)

# ── 8.1  Prepare feature matrix X and target vector y ────────────────────────
X_ling = df_clean[FEATURE_COLS].values.astype(float)
y      = df_clean["helpful"].values

print(f"\n  Feature matrix shape : {X_ling.shape}")
print(f"  Target vector shape  : {y.shape}")
print(f"  Class balance        : {y.mean()*100:.1f}% helpful")

# ── 8.2  Train / test split (stratified to preserve class balance) ────────────
X_train_l, X_test_l, y_train, y_test = train_test_split(
    X_ling, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y           # ensures same class ratio in train & test
)

print(f"\n  Train set : {X_train_l.shape[0]:,} rows")
print(f"  Test set  : {X_test_l.shape[0]:,} rows")
print(f"  Train helpful % : {y_train.mean()*100:.1f}%")
print(f"  Test  helpful % : {y_test.mean()*100:.1f}%")

# ── 8.3  Scale features (required for Logistic Regression) ───────────────────
# StandardScaler: zero mean, unit variance on training data only.
# Fitted on train, applied to test — prevents data leakage.
scaler_ling  = StandardScaler()
X_train_ls   = scaler_ling.fit_transform(X_train_l)
X_test_ls    = scaler_ling.transform(X_test_l)

# ── 8.4  Train Logistic Regression ───────────────────────────────────────────
# C=1.0  : default regularisation strength (L2 penalty)
# max_iter=1000 : ensure convergence on scaled features
lr_ling = LogisticRegression(
    C=1.0,
    penalty="l2",
    solver="lbfgs",
    max_iter=1000,
    class_weight="balanced",   # corrects for class imbalance
    random_state=RANDOM_STATE
)
lr_ling.fit(X_train_ls, y_train)
print(f"\n  Logistic Regression trained")

# ── 8.5  Predictions ─────────────────────────────────────────────────────────
y_pred_l  = lr_ling.predict(X_test_ls)
y_prob_l  = lr_ling.predict_proba(X_test_ls)[:, 1]   # probability of class 1

# ── 8.6  Classification report ───────────────────────────────────────────────
print(f"\n  ── Classification Report ──")
print(classification_report(
    y_test, y_pred_l,
    target_names=["Not Helpful (0)", "Helpful (1)"]
))

# ── 8.7  ROC-AUC ─────────────────────────────────────────────────────────────
roc_auc_l = roc_auc_score(y_test, y_prob_l)
print(f"  ROC-AUC : {roc_auc_l:.4f}")

# ── 8.8  5-fold stratified cross-validation ──────────────────────────────────
# Cross-validation on the FULL scaled dataset gives a more robust
# estimate of generalisation than a single train/test split.
print(f"\n  Running 5-fold cross-validation (F1-macro)...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Scale full feature matrix for CV
X_ling_scaled_full = scaler_ling.fit_transform(X_ling)

cv_scores_l = cross_val_score(
    LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    X_ling_scaled_full, y,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1
)
print(f"  CV F1-macro scores : {[round(s,4) for s in cv_scores_l]}")
print(f"  Mean CV F1-macro   : {cv_scores_l.mean():.4f} ± {cv_scores_l.std():.4f}")

# ── 8.9  Figure 13: ROC curve ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(
    y_test, y_prob_l,
    name=f"LR — Linguistic Only  (AUC = {roc_auc_l:.3f})",
    color="#4C72B0",
    ax=ax
)
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random classifier")
ax.set_title("ROC Curve — Model 1: Logistic Regression (Linguistic Features)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("fig13_roc_model1.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n  Saved: fig13_roc_model1.png")

# ── 8.10 Figure 14: Confusion matrix ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_l,
    display_labels=["Not Helpful", "Helpful"],
    colorbar=False,
    cmap="Blues",
    ax=ax
)
ax.set_title("Confusion Matrix — Model 1: LR (Linguistic Features)",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("fig14_cm_model1.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig14_cm_model1.png")

# ── 8.11 Figure 15: Logistic Regression coefficients ─────────────────────────

coef_series = pd.Series(
    lr_ling.coef_[0],
    index=[c.replace("feat_","").replace("_"," ").title() for c in FEATURE_COLS]
).sort_values()

colors_coef = ["#DD4949" if v < 0 else "#2ca02c" for v in coef_series.values]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(coef_series.index, coef_series.values,
               color=colors_coef, alpha=0.85, edgecolor="white")

for bar, val in zip(bars, coef_series.values):
    x_pos = val + 0.005 if val >= 0 else val - 0.005
    ha    = "left"       if val >= 0 else "right"
    ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
            f"{val:+.3f}", va="center", ha=ha, fontsize=8.5)

ax.axvline(0, color="black", linewidth=0.9)
ax.set_title("Logistic Regression Coefficients\n"
             "(Model 1: Linguistic Features Only — standardised)",
             fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Coefficient (log-odds of being Helpful)")

legend_elements = [
    Patch(facecolor="#2ca02c", label="Increases helpfulness probability"),
    Patch(facecolor="#DD4949", label="Decreases helpfulness probability")
]
ax.legend(handles=legend_elements, fontsize=9)
plt.tight_layout()
plt.savefig("fig15_lr_coefficients.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig15_lr_coefficients.png")

# ── 8.12 Store Model 1 results for final comparison table (Steps 9–12) ───────
# We build a results dictionary that all model steps will append to.
model_results = []
model_results.append({
    "model_name"   : "LR — Linguistic Only",
    "accuracy"     : (y_pred_l == y_test).mean(),
    "f1_macro"     : float(classification_report(y_test, y_pred_l,
                           output_dict=True)["macro avg"]["f1-score"]),
    "f1_helpful"   : float(classification_report(y_test, y_pred_l,
                           output_dict=True)["1"]["f1-score"]),
    "roc_auc"      : roc_auc_l,
    "cv_f1_mean"   : cv_scores_l.mean(),
    "cv_f1_std"    : cv_scores_l.std(),
    "y_prob"       : y_prob_l,
    "y_pred"       : y_pred_l,
})

print(f"\n  ── Model 1 Summary ──")
print(f"  Accuracy     : {model_results[0]['accuracy']:.4f}")
print(f"  F1-macro     : {model_results[0]['f1_macro']:.4f}")
print(f"  F1-helpful   : {model_results[0]['f1_helpful']:.4f}")
print(f"  ROC-AUC      : {model_results[0]['roc_auc']:.4f}")
print(f"  CV F1-macro  : {model_results[0]['cv_f1_mean']:.4f} "
      f"± {model_results[0]['cv_f1_std']:.4f}")

print("\n" + "=" * 65)
print("  STEP 8 COMPLETE — Model 1: LR (Linguistic Features Only)")
print("=" * 65)
print("  Figures saved:")
for f in ["fig13_roc_model1.png", "fig14_cm_model1.png",
          "fig15_lr_coefficients.png"]:
    print(f"    - {f}")
print("  Model 1 results stored in: model_results[0]")
print("  ➡  Next: Step 9 — Model 2: Logistic Regression (TF-IDF + Linguistic)")


# =============================================================================
# STEP 9 — MODEL 2: LOGISTIC REGRESSION (TF-IDF + LINGUISTIC FEATURES)


print("\n" + "=" * 65)
print("   Model 2: Logistic Regression (TF-IDF + Linguistic)")
print("=" * 65)

# ── 9.1  Build TF-IDF matrix ─────────────────────────────────────────────────
print("\n  Fitting TF-IDF vectoriser...")

tfidf_vec = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=5,
    stop_words="english"
)

texts_all = df_clean["Text"].fillna("").astype(str)

# Fit on training indices only to prevent leakage
train_idx = df_clean.index[
    df_clean.index.isin(
        pd.Series(range(len(df_clean))).sample(
            frac=0.8, random_state=RANDOM_STATE
        )
    )
]

# Re-derive the same train/test split as Step 8 using indices
n = len(df_clean)
idx = np.arange(n)
idx_train, idx_test = train_test_split(idx, test_size=0.2,
                          random_state=RANDOM_STATE, stratify=y)

X_tfidf_train = tfidf_vec.fit_transform(texts_all.iloc[idx_train])
X_tfidf_test  = tfidf_vec.transform(texts_all.iloc[idx_test])

print(f"  TF-IDF train matrix : {X_tfidf_train.shape}")
print(f"  TF-IDF test  matrix : {X_tfidf_test.shape}")
print(f"  Vocabulary size     : {len(tfidf_vec.vocabulary_):,} terms")

# ── 9.2  Scale linguistic features (same split as Step 8) ────────────────────
X_ling_train_raw = X_ling[idx_train]
X_ling_test_raw  = X_ling[idx_test]

scaler_combined   = StandardScaler()
X_ling_train_sc   = scaler_combined.fit_transform(X_ling_train_raw)
X_ling_test_sc    = scaler_combined.transform(X_ling_test_raw)

y_train_c = y[idx_train]
y_test_c  = y[idx_test]

# ── 9.3  Combine TF-IDF (sparse) + linguistic (dense) into one matrix ────────
X_train_comb = sp.hstack([X_tfidf_train, sp.csr_matrix(X_ling_train_sc)])
X_test_comb  = sp.hstack([X_tfidf_test,  sp.csr_matrix(X_ling_test_sc)])

print(f"\n  Combined train matrix : {X_train_comb.shape}")
print(f"  Combined test  matrix : {X_test_comb.shape}")

# ── 9.4  Train Logistic Regression on combined features ──────────────────────
lr_comb = LogisticRegression(
    C=1.0, penalty="l2", solver="saga",
    max_iter=1000, class_weight="balanced",
    random_state=RANDOM_STATE, n_jobs=-1
)
lr_comb.fit(X_train_comb, y_train_c)
print(f"\n  ✅ Logistic Regression (combined) trained")

# ── 9.5  Evaluate ─────────────────────────────────────────────────────────────
y_pred_c = lr_comb.predict(X_test_comb)
y_prob_c = lr_comb.predict_proba(X_test_comb)[:, 1]

print(f"\n  ── Classification Report ──")
print(classification_report(y_test_c, y_pred_c,
      target_names=["Not Helpful (0)", "Helpful (1)"]))

roc_auc_c = roc_auc_score(y_test_c, y_prob_c)
print(f"  ROC-AUC : {roc_auc_c:.4f}")

# ── 9.6  Figure 16: ROC curve Model 2 ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(
    y_test_c, y_prob_c,
    name=f"LR — TF-IDF + Linguistic  (AUC = {roc_auc_c:.3f})",
    color="#DD8452", ax=ax
)
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random classifier")
ax.set_title("ROC Curve — Model 2: LR (TF-IDF + Linguistic)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("fig16_roc_model2.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n  Saved: fig16_roc_model2.png")

# ── 9.7  Figure 17: Confusion matrix Model 2 ─────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test_c, y_pred_c,
    display_labels=["Not Helpful", "Helpful"],
    colorbar=False, cmap="Oranges", ax=ax
)
ax.set_title("Confusion Matrix — Model 2: LR (TF-IDF + Linguistic)",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("fig17_cm_model2.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig17_cm_model2.png")

# ── 9.8  Store Model 2 results ────────────────────────────────────────────────
report_c = classification_report(y_test_c, y_pred_c, output_dict=True)
model_results.append({
    "model_name" : "LR — TF-IDF + Linguistic",
    "accuracy"   : report_c["accuracy"],
    "f1_macro"   : report_c["macro avg"]["f1-score"],
    "f1_helpful" : report_c["1"]["f1-score"],
    "roc_auc"    : roc_auc_c,
    "cv_f1_mean" : None,   # CV skipped for TF-IDF model (cost vs benefit)
    "cv_f1_std"  : None,
    "y_prob"     : y_prob_c,
    "y_pred"     : y_pred_c,
})

print(f"\n  ── Model 2 Summary ──")
print(f"  Accuracy   : {report_c['accuracy']:.4f}")
print(f"  F1-macro   : {report_c['macro avg']['f1-score']:.4f}")
print(f"  ROC-AUC    : {roc_auc_c:.4f}")
print(f"  Δ AUC vs Model 1: {roc_auc_c - model_results[0]['roc_auc']:+.4f}")

print("\n" + "=" * 65)

print("=" * 65)



# =============================================================================
# STEP 10 — MODEL 3: RANDOM FOREST (TF-IDF + LINGUISTIC FEATURES)


print("\n" + "=" * 65)
print("   Model 3: Random Forest (TF-IDF + Linguistic)")
print("=" * 65)

# ── 10.1  Train Random Forest (reuses X_train_comb / X_test_comb from Step 9) ─
print("\n  Training Random Forest (200 trees)...")
print("  (This may take 2–4 minutes)")

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=5,
    max_features="sqrt",      # √(n_features) per split — standard for RF
    class_weight="balanced",  # prevents majority-class collapse
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf.fit(X_train_comb, y_train_c)
print(f"   Random Forest trained")

# ── 10.2  Evaluate ────────────────────────────────────────────────────────────
y_pred_rf = rf.predict(X_test_comb)
y_prob_rf = rf.predict_proba(X_test_comb)[:, 1]

print(f"\n  ── Classification Report ──")
print(classification_report(y_test_c, y_pred_rf,
      target_names=["Not Helpful (0)", "Helpful (1)"]))

roc_auc_rf = roc_auc_score(y_test_c, y_prob_rf)
print(f"  ROC-AUC : {roc_auc_rf:.4f}")

# ── 10.3  Figure 18: ROC curve Model 3 ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(
    y_test_c, y_prob_rf,
    name=f"Random Forest  (AUC = {roc_auc_rf:.3f})",
    color="#55A868", ax=ax
)
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random classifier")
ax.set_title("ROC Curve — Model 3: Random Forest (TF-IDF + Linguistic)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("fig18_roc_model3.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n  Saved: fig18_roc_model3.png")

# ── 10.4  Figure 19: Confusion matrix Model 3 ────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test_c, y_pred_rf,
    display_labels=["Not Helpful", "Helpful"],
    colorbar=False, cmap="Greens", ax=ax
)
ax.set_title("Confusion Matrix — Model 3: Random Forest",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("fig19_cm_model3.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig19_cm_model3.png")

# ── 10.5  Figure 20: Random Forest feature importances (linguistic only) ─────
# Extract importances for the LINGUISTIC features only
# (they are the last len(FEATURE_COLS) columns in the combined matrix)
n_tfidf   = X_tfidf_train.shape[1]
rf_importances = rf.feature_importances_

# Linguistic feature importances (columns after TF-IDF block)
ling_importances = rf_importances[n_tfidf:]
feat_imp_series  = pd.Series(
    ling_importances,
    index=[c.replace("feat_","").replace("_"," ").title() for c in FEATURE_COLS]
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors_imp = ["#2ca02c" if v >= feat_imp_series.median()
              else "#4C72B0" for v in feat_imp_series.values]
feat_imp_series.plot(kind="barh", ax=ax, color=colors_imp, alpha=0.85,
                     edgecolor="white")
ax.set_title("Random Forest: Linguistic Feature Importances\n"
             "(contribution from the 17 handcrafted features)",
             fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Mean Decrease in Impurity (Gini)")
ax.axvline(feat_imp_series.median(), color="red", linestyle="--",
           linewidth=1, label=f"Median = {feat_imp_series.median():.4f}")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("fig20_rf_feature_importance.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig20_rf_feature_importance.png")

print(f"\n  Top 5 linguistic features (Random Forest):")
for feat, imp in feat_imp_series.sort_values(ascending=False).head(5).items():
    print(f"    {feat:<35} {imp:.5f}")

# ── 10.6  Store Model 3 results ───────────────────────────────────────────────
report_rf = classification_report(y_test_c, y_pred_rf, output_dict=True)
model_results.append({
    "model_name" : "Random Forest — TF-IDF + Linguistic",
    "accuracy"   : report_rf["accuracy"],
    "f1_macro"   : report_rf["macro avg"]["f1-score"],
    "f1_helpful" : report_rf["1"]["f1-score"],
    "roc_auc"    : roc_auc_rf,
    "cv_f1_mean" : None,
    "cv_f1_std"  : None,
    "y_prob"     : y_prob_rf,
    "y_pred"     : y_pred_rf,
})

print(f"\n  ── Model 3 Summary ──")
print(f"  Accuracy   : {report_rf['accuracy']:.4f}")
print(f"  F1-macro   : {report_rf['macro avg']['f1-score']:.4f}")
print(f"  ROC-AUC    : {roc_auc_rf:.4f}")
print(f"  Δ AUC vs Model 1 : {roc_auc_rf - model_results[0]['roc_auc']:+.4f}")
print(f"  Δ AUC vs Model 2 : {roc_auc_rf - model_results[1]['roc_auc']:+.4f}")

print("\n" + "=" * 65)
print("   — Model 3: Random Forest")
print("=" * 65)



# =============================================================================
# STEP 11 — MODEL 4: GRADIENT BOOSTING (TF-IDF + LINGUISTIC FEATURES)
# =============================================================================


print("\n" + "=" * 65)
print(" Model 4: Gradient Boosting (TF-IDF + Linguistic)")
print("=" * 65)

# ── 11.1  Convert sparse matrix to dense array for GBM ───────────────────────
# GradientBoostingClassifier does not natively support sparse matrices,
# so we use a dense representation. To keep memory manageable we reduce
# the TF-IDF vocabulary to 3,000 features for this model only.
print("\n  Preparing dense feature matrix for Gradient Boosting...")
print("  (Reducing TF-IDF to 3,000 features for memory efficiency)")

tfidf_gb = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=5,
    stop_words="english"
)

X_tfidf_gb_train = tfidf_gb.fit_transform(texts_all.iloc[idx_train]).toarray()
X_tfidf_gb_test  = tfidf_gb.transform(texts_all.iloc[idx_test]).toarray()

# Re-scale linguistic features (same scaler)
X_ling_gb_train = scaler_combined.transform(X_ling_train_raw)
X_ling_gb_test  = scaler_combined.transform(X_ling_test_raw)

# Concatenate TF-IDF (dense) + linguistic features
X_train_gb = np.hstack([X_tfidf_gb_train, X_ling_gb_train])
X_test_gb  = np.hstack([X_tfidf_gb_test,  X_ling_gb_test])

print(f"  GBM train matrix : {X_train_gb.shape}")
print(f"  GBM test  matrix : {X_test_gb.shape}")

# ── 11.2  Train Gradient Boosting ─────────────────────────────────────────────
print("\n  Training Gradient Boosting (150 estimators)...")
print("  (This may take 3–6 minutes)")

gb = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.08,
    subsample=0.8,
    min_samples_leaf=10,
    random_state=RANDOM_STATE
)
gb.fit(X_train_gb, y_train_c)
print(f"   Gradient Boosting trained")

# ── 11.3  Evaluate ────────────────────────────────────────────────────────────
y_pred_gb = gb.predict(X_test_gb)
y_prob_gb = gb.predict_proba(X_test_gb)[:, 1]

print(f"\n  ── Classification Report ──")
print(classification_report(y_test_c, y_pred_gb,
      target_names=["Not Helpful (0)", "Helpful (1)"]))

roc_auc_gb = roc_auc_score(y_test_c, y_prob_gb)
print(f"  ROC-AUC : {roc_auc_gb:.4f}")

# ── 11.4  Figure 21: ROC curve Model 4 ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(
    y_test_c, y_prob_gb,
    name=f"Gradient Boosting  (AUC = {roc_auc_gb:.3f})",
    color="#C44E52", ax=ax
)
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random classifier")
ax.set_title("ROC Curve — Model 4: Gradient Boosting (TF-IDF + Linguistic)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("fig21_roc_model4.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n  Saved: fig21_roc_model4.png")

# ── 11.5  Figure 22: Confusion matrix Model 4 ────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test_c, y_pred_gb,
    display_labels=["Not Helpful", "Helpful"],
    colorbar=False, cmap="Reds", ax=ax
)
ax.set_title("Confusion Matrix — Model 4: Gradient Boosting",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("fig22_cm_model4.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig22_cm_model4.png")

# ── 11.6  Figure 23: GBM feature importances (linguistic only) ───────────────
n_tfidf_gb        = X_tfidf_gb_train.shape[1]
gb_ling_imp       = gb.feature_importances_[n_tfidf_gb:]
gb_imp_series     = pd.Series(
    gb_ling_imp,
    index=[c.replace("feat_","").replace("_"," ").title() for c in FEATURE_COLS]
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors_gb = ["#C44E52" if v >= gb_imp_series.median()
             else "#8C8C8C" for v in gb_imp_series.values]
gb_imp_series.plot(kind="barh", ax=ax, color=colors_gb,
                   alpha=0.85, edgecolor="white")
ax.set_title("Gradient Boosting: Linguistic Feature Importances",
             fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Feature Importance Score")
ax.axvline(gb_imp_series.median(), color="navy", linestyle="--",
           linewidth=1, label=f"Median = {gb_imp_series.median():.4f}")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("fig23_gb_feature_importance.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig23_gb_feature_importance.png")

print(f"\n  Top 5 linguistic features (Gradient Boosting):")
for feat, imp in gb_imp_series.sort_values(ascending=False).head(5).items():
    print(f"    {feat:<35} {imp:.5f}")

# ── 11.7  Store Model 4 results ───────────────────────────────────────────────
report_gb = classification_report(y_test_c, y_pred_gb, output_dict=True)
model_results.append({
    "model_name" : "Gradient Boosting — TF-IDF + Linguistic",
    "accuracy"   : report_gb["accuracy"],
    "f1_macro"   : report_gb["macro avg"]["f1-score"],
    "f1_helpful" : report_gb["1"]["f1-score"],
    "roc_auc"    : roc_auc_gb,
    "cv_f1_mean" : None,
    "cv_f1_std"  : None,
    "y_prob"     : y_prob_gb,
    "y_pred"     : y_pred_gb,
})

print(f"\n   Model 4 Summary ──")
print(f"  Accuracy   : {report_gb['accuracy']:.4f}")
print(f"  F1-macro   : {report_gb['macro avg']['f1-score']:.4f}")
print(f"  ROC-AUC    : {roc_auc_gb:.4f}")
print(f"  Δ AUC vs Model 1 : {roc_auc_gb - model_results[0]['roc_auc']:+.4f}")
print(f"  Δ AUC vs Model 2 : {roc_auc_gb - model_results[1]['roc_auc']:+.4f}")
print(f"  Δ AUC vs Model 3 : {roc_auc_gb - model_results[2]['roc_auc']:+.4f}")

print("\n" + "=" * 65)
print("  STEP 11 COMPLETE — Model 4: Gradient Boosting")
print("=" * 65)
print("  Figures saved:")
for f in ["fig16_roc_model2.png",  "fig17_cm_model2.png",
          "fig18_roc_model3.png",  "fig19_cm_model3.png",
          "fig20_rf_feature_importance.png",
          "fig21_roc_model4.png",  "fig22_cm_model4.png",
          "fig23_gb_feature_importance.png"]:
    print(f"    - {f}")
print(f"\n  All 4 models stored in model_results ({len(model_results)} entries)")



# =============================================================================
# STEP 12 — FINAL MODEL SELECTION, COMPARISON & INTERPRETATION


print("\n" + "=" * 65)
print("   Final Model Selection & Comparison")
print("=" * 65)

# ── 12.1  Build comparison dataframe ─────────────────────────────────────────
results_df = pd.DataFrame([
    {
        "Model"      : r["model_name"],
        "Accuracy"   : round(r["accuracy"],   4),
        "F1-Macro"   : round(r["f1_macro"],   4),
        "F1-Helpful" : round(r["f1_helpful"], 4),
        "ROC-AUC"    : round(r["roc_auc"],    4),
    }
    for r in model_results
])

print(f"\n  ── Full Model Comparison Table ──")
print(results_df.to_string(index=False))

# Identify best model by ROC-AUC
best_idx   = results_df["ROC-AUC"].idxmax()
best_row   = results_df.loc[best_idx]
best_result = model_results[best_idx]

print(f"\n  🏆 Best model (by ROC-AUC): {best_row['Model']}")
print(f"     ROC-AUC    : {best_row['ROC-AUC']:.4f}")
print(f"     Accuracy   : {best_row['Accuracy']:.4f}")
print(f"     F1-Macro   : {best_row['F1-Macro']:.4f}")
print(f"     F1-Helpful : {best_row['F1-Helpful']:.4f}")

# ── 12.2  Figure 24: Model comparison bar chart ───────────────────────────────
metrics     = ["Accuracy", "F1-Macro", "F1-Helpful", "ROC-AUC"]
metric_colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
x           = np.arange(len(results_df))
bar_width   = 0.2

fig, ax = plt.subplots(figsize=(15, 7))

for i, (metric, color) in enumerate(zip(metrics, metric_colors)):
    bars = ax.bar(
        x + i * bar_width,
        results_df[metric],
        width=bar_width,
        label=metric,
        color=color,
        alpha=0.85,
        edgecolor="white"
    )
    # Annotate bars
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f"{bar.get_height():.3f}",
            ha="center", va="bottom", fontsize=7.5, fontweight="bold"
        )

# Highlight best model column
ax.axvspan(best_idx - 0.15, best_idx + 4 * bar_width - 0.05,
           alpha=0.08, color="gold", zorder=0)
ax.text(best_idx + bar_width, results_df[metrics].values.max() + 0.025,
        "🏆 Best", ha="center", fontsize=11, color="goldenrod",
        fontweight="bold")

ax.set_xticks(x + bar_width * 1.5)
ax.set_xticklabels(results_df["Model"], fontsize=9)
ax.set_ylim(results_df[metrics].values.min() - 0.05,
            results_df[metrics].values.max() + 0.06)
ax.set_ylabel("Score")
ax.set_title("Model Comparison: All 4 Models × 4 Metrics",
             fontsize=14, fontweight="bold", pad=14)
ax.legend(fontsize=10, loc="lower right")
ax.axhline(0.5, color="gray", linewidth=0.8, linestyle=":",
           label="Chance level")

plt.tight_layout()
plt.savefig("fig24_model_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n  Saved: fig24_model_comparison.png")

# ── 12.3  Figure 25: Overlaid ROC curves for all 4 models ────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
colors_roc = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

for r, color in zip(model_results, colors_roc):
    RocCurveDisplay.from_predictions(
        y_test_c,
        r["y_prob"],
        name=f"{r['model_name']}  (AUC={r['roc_auc']:.3f})",
        color=color,
        ax=ax,
        alpha=0.85
    )

ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="Random classifier")
ax.set_title("Overlaid ROC Curves — All 4 Models",
             fontsize=13, fontweight="bold", pad=12)
ax.legend(fontsize=8.5, loc="lower right")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
plt.tight_layout()
plt.savefig("fig25_roc_all_models.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig25_roc_all_models.png")

# ── 12.4  Figure 26: Confusion matrix of best model ──────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test_c,
    best_result["y_pred"],
    display_labels=["Not Helpful", "Helpful"],
    colorbar=False,
    cmap="YlOrRd",
    ax=ax
)
ax.set_title(f"Confusion Matrix — Best Model\n({best_row['Model']})",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("fig26_cm_best_model.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig26_cm_best_model.png")

# ── 12.5  Figure 27: Top TF-IDF terms from best LR model (if applicable) ─────

if "LR" in best_row["Model"] and "TF-IDF" in best_row["Model"]:
    feature_names_tfidf = np.array(tfidf_vec.get_feature_names_out())
    coef_all            = lr_comb.coef_[0]

    # TF-IDF coefficients only (first n_tfidf columns)
    coef_tfidf = coef_all[:len(feature_names_tfidf)]

    top_pos_idx = coef_tfidf.argsort()[-20:][::-1]
    top_neg_idx = coef_tfidf.argsort()[:20]

    top_pos = pd.Series(coef_tfidf[top_pos_idx],
                        index=feature_names_tfidf[top_pos_idx])
    top_neg = pd.Series(coef_tfidf[top_neg_idx],
                        index=feature_names_tfidf[top_neg_idx])

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle("Top 20 TF-IDF Term Weights — Best Model (LR)\n"
                 "Positive = pushes toward Helpful | Negative = pushes toward Not Helpful",
                 fontsize=12, fontweight="bold")

    top_pos.sort_values().plot(kind="barh", ax=axes[0],
                               color="#2ca02c", alpha=0.85, edgecolor="white")
    axes[0].set_title("Top 20 Terms → Helpful", fontsize=11, fontweight="bold")
    axes[0].set_xlabel("LR Coefficient (log-odds)")

    top_neg.sort_values(ascending=False).plot(kind="barh", ax=axes[1],
                                              color="#DD4949", alpha=0.85,
                                              edgecolor="white")
    axes[1].set_title("Top 20 Terms → Not Helpful", fontsize=11, fontweight="bold")
    axes[1].set_xlabel("LR Coefficient (log-odds)")

    plt.tight_layout()
    plt.savefig("fig27_tfidf_term_weights.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("  Saved: fig27_tfidf_term_weights.png")
else:
    print("  (Fig 27 skipped — best model is not LR+TF-IDF)")

# ── 12.6  Model selection justification (printed for paper) ──────────────────


print("\n" + "=" * 65)
print("  STEP 12 COMPLETE — Final Model Selection & Comparison")
print("=" * 65)
print(f"  Best model : {best_row['Model']}")
print(f"  ROC-AUC    : {best_row['ROC-AUC']:.4f}")
print("  ➡  Next: Step 13 — Sentiment Deep Dive & Bias Discussion")


# =============================================================================
# STEP 13 — SENTIMENT DEEP DIVE & BIAS DISCUSSION
# =============================================================================


print("\n" + "=" * 65)
print("  Sentiment Deep Dive & Bias Discussion")
print("=" * 65)

# ── 13.1  Figure 28: VADER sentiment distributions by class ──────────────────
print("\n  Plotting sentiment distributions...")

sentiment_feats  = ["feat_sentiment_compound",
                    "feat_sentiment_pos",
                    "feat_sentiment_neg",
                    "feat_sentiment_neu"]
sentiment_titles = ["Compound Score\n(overall: -1 negative → +1 positive)",
                    "Positive Score\n(proportion of positive language)",
                    "Negative Score\n(proportion of negative language)",
                    "Neutral Score\n(proportion of neutral language)"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("VADER Sentiment Score Distributions by Helpfulness Label",
             fontsize=14, fontweight="bold")
axes = axes.flatten()

for ax, feat, title in zip(axes, sentiment_feats, sentiment_titles):
    for label_val, color, name in [(0, "#DD4949", "Not Helpful"),
                                    (1, "#2ca02c", "Helpful")]:
        subset = df_clean[df_clean["helpful"] == label_val][feat]
        subset.plot(kind="kde", ax=ax, color=color, linewidth=2,
                    label=f"{name}  (μ={subset.mean():.3f})")

    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlabel("VADER Score")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8.5)
    ax.axvline(df_clean[feat].mean(), color="grey",
               linewidth=1, linestyle=":", alpha=0.7)

plt.tight_layout()
plt.savefig("fig28_sentiment_distributions.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig28_sentiment_distributions.png")

# ── 13.2  Figure 29: Sentiment vs helpfulness ratio scatter ──────────────────
print("  Plotting sentiment vs helpfulness scatter...")

scatter_s = df_clean.sample(min(3000, len(df_clean)),
                             random_state=RANDOM_STATE)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("VADER Sentiment vs Helpfulness Ratio",
             fontsize=13, fontweight="bold")

# Left: compound sentiment vs ratio, coloured by helpful label
sc1 = axes[0].scatter(
    scatter_s["feat_sentiment_compound"],
    scatter_s["helpfulness_ratio"],
    c=scatter_s["helpful"].map({0: "#DD4949", 1: "#2ca02c"}),
    alpha=0.35, s=18, edgecolors="none"
)
axes[0].axhline(HELPFULNESS_THRESHOLD, color="navy",
                linewidth=1.2, linestyle="--",
                label=f"Threshold ({HELPFULNESS_THRESHOLD})")
axes[0].set_title("Compound Sentiment vs Helpfulness Ratio")
axes[0].set_xlabel("VADER Compound Score")
axes[0].set_ylabel("Helpfulness Ratio")

legend_e = [Patch(facecolor="#2ca02c", alpha=0.7, label="Helpful"),
            Patch(facecolor="#DD4949", alpha=0.7, label="Not Helpful")]
axes[0].legend(handles=legend_e, fontsize=9)

# Right: compound sentiment vs ratio, coloured by star rating
cmap  = plt.cm.RdYlGn
norm  = plt.Normalize(vmin=1, vmax=5)
sc2   = axes[1].scatter(
    scatter_s["feat_sentiment_compound"],
    scatter_s["helpfulness_ratio"],
    c=scatter_s["Score"], cmap=cmap, norm=norm,
    alpha=0.4, s=18, edgecolors="none"
)
axes[1].axhline(HELPFULNESS_THRESHOLD, color="navy",
                linewidth=1.2, linestyle="--",
                label=f"Threshold ({HELPFULNESS_THRESHOLD})")
plt.colorbar(sc2, ax=axes[1], label="Star Rating (1–5)")
axes[1].set_title("Compound Sentiment vs Helpfulness Ratio\n(coloured by Star Rating)")
axes[1].set_xlabel("VADER Compound Score")
axes[1].set_ylabel("Helpfulness Ratio")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig("fig29_sentiment_vs_helpfulness.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig29_sentiment_vs_helpfulness.png")

# ── 13.3  Figure 30: Mean sentiment heatmap — Star × Label ───────────────────
print("  Plotting sentiment heatmap...")

heatmap_data = (
    df_clean.groupby(["Score", "helpful"])["feat_sentiment_compound"]
    .mean()
    .unstack()
    .rename(columns={0: "Not Helpful", 1: "Helpful"})
)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    heatmap_data, annot=True, fmt=".3f",
    cmap="RdYlGn", center=0,
    linewidths=0.5, linecolor="white",
    ax=ax, cbar_kws={"label": "Mean VADER Compound Score"}
)
ax.set_title("Mean VADER Compound Sentiment\nby Star Rating × Helpfulness Label",
             fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Helpfulness Label")
ax.set_ylabel("Star Rating")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.savefig("fig30_sentiment_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig30_sentiment_heatmap.png")

# ── 13.4  Print sentiment summary statistics ──────────────────────────────────
print(f"\n  ── Sentiment Summary: Helpful vs Not Helpful ──")
sent_summary = df_clean.groupby("helpful")[sentiment_feats].mean()
sent_summary.index = ["Not Helpful (0)", "Helpful (1)"]
sent_summary.columns = ["Compound", "Positive", "Negative", "Neutral"]
print(sent_summary.to_string())

# ─

print("\n" + "=" * 65)
print("   Sentiment Deep Dive & Bias Discussion")
print("=" * 65)
print("  Figures saved:")
for f in ["fig28_sentiment_distributions.png",
          "fig29_sentiment_vs_helpfulness.png",
          "fig30_sentiment_heatmap.png"]:
    print(f"    - {f}")



# =============================================================================
# STEP 14 — REPLICATION SUMMARY & FINAL OUTPUT


print("\n" + "=" * 65)
print("  STEP 14 — Replication Summary & Final Output")
print("=" * 65)

# ── 14.1  All figures generated ───────────────────────────────────────────────
all_figures = [
    ("fig1_helpfulness_ratio.png",       "Helpfulness ratio distribution"),
    ("fig2_class_balance.png",           "Class balance bar + pie chart"),
    ("fig3_star_ratings.png",            "Star rating distributions"),
    ("fig4_review_length.png",           "Review length by label"),
    ("fig5_wordclouds.png",              "Word clouds: helpful vs not helpful"),
    ("fig6_top_words.png",               "Top 20 word frequencies by class"),
    ("fig7_tfidf_words.png",             "Top 20 TF-IDF distinctive words"),
    ("fig8_punctuation_patterns.png",    "Punctuation & sentence structure"),
    ("fig9_feature_label_correlation.png","Feature–label correlations"),
    ("fig10_violin_plots.png",           "Violin plots: top 6 features"),
    ("fig11_feature_correlation_heatmap.png", "Feature–feature heatmap"),
    ("fig12_wordcount_vs_helpfulness.png","Word count vs helpfulness scatter"),
    ("fig13_roc_model1.png",             "ROC — Model 1: LR Linguistic"),
    ("fig14_cm_model1.png",              "CM  — Model 1: LR Linguistic"),
    ("fig15_lr_coefficients.png",        "LR coefficients (linguistic)"),
    ("fig16_roc_model2.png",             "ROC — Model 2: LR TF-IDF+Ling"),
    ("fig17_cm_model2.png",              "CM  — Model 2: LR TF-IDF+Ling"),
    ("fig18_roc_model3.png",             "ROC — Model 3: Random Forest"),
    ("fig19_cm_model3.png",              "CM  — Model 3: Random Forest"),
    ("fig20_rf_feature_importance.png",  "RF feature importances"),
    ("fig21_roc_model4.png",             "ROC — Model 4: Gradient Boosting"),
    ("fig22_cm_model4.png",              "CM  — Model 4: Gradient Boosting"),
    ("fig23_gb_feature_importance.png",  "GBM feature importances"),
    ("fig24_model_comparison.png",       "All models × all metrics bar chart"),
    ("fig25_roc_all_models.png",         "Overlaid ROC curves all 4 models"),
    ("fig26_cm_best_model.png",          "CM  — Best model"),
    ("fig27_tfidf_term_weights.png",     "Top TF-IDF term weights (best LR)"),
    ("fig28_sentiment_distributions.png","VADER distributions by class"),
    ("fig29_sentiment_vs_helpfulness.png","Sentiment vs helpfulness scatter"),
    ("fig30_sentiment_heatmap.png",      "Sentiment heatmap star × label"),
]

print(f"\n  ── All {len(all_figures)} Figures Generated ──")
for fname, desc in all_figures:
    print(f"  {fname:<45}  {desc}")

# ── 14.2  Final model performance table 
print(f"\n  ── Final Model Performance Table (copy into paper) ──")
print(f"\n  {'Model':<42} {'Acc':>6} {'F1-Mac':>7} {'F1-Help':>8} {'AUC':>7}")
print("  " + "─" * 75)
for r in model_results:
    marker = " ◄ BEST" if r["model_name"] == best_row["Model"] else ""
    print(f"  {r['model_name']:<42} "
          f"{r['accuracy']:>6.4f} "
          f"{r['f1_macro']:>7.4f} "
          f"{r['f1_helpful']:>8.4f} "
          f"{r['roc_auc']:>7.4f}"
          f"{marker}")


  Random state : 42
  Stopwords    : 198 English words loaded
  VADER        : ready

   Data Loading & Inspection

  Loading: Reviews.csv
  Raw shape       : 568,454 rows × 10 columns

  ── Column names & dtypes ──
Id                         int64
ProductId                 object
UserId                    object
ProfileName               object
HelpfulnessNumerator       int64
HelpfulnessDenominator     int64
Score                      int64
Time                       int64
Summary                   object
Text                      object

  ── First 3 rows (selected columns) ──
 Id  ProductId  Score  HelpfulnessNumerator  HelpfulnessDenominator               Summary                                                                                                                                                                                                                                                                                                                                    